# Naive Bayesian Classifier

## Objective
Implement the **Multinomial Naive Bayes** algorithm from scratch using **NumPy** and compare its performance with Scikit-learn's implementation on a spam email dataset.

### Concepts Covered
- Bayes Theorem
- Conditional Probability
- Independence Assumption
- Laplace Smoothing
- Text Classification

In [1]:
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

## Load Dataset

In [2]:
df = pd.read_csv('spam_ham_dataset.csv')

In [3]:
df.head()

,Unnamed: 0,label,text,label_num
0,605,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
1,2349,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,3624,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,4685,spam,"Subject: photoshop , windows , office . cheap ...",1
4,2030,ham,Subject: re : indian springs\r\nthis deal is t...,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5171 entries, 0 to 5170
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  5171 non-null   int64 
 1   label       5171 non-null   object
 2   text        5171 non-null   object
 3   label_num   5171 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 161.7+ KB


In [5]:
df = df.drop(columns=["Unnamed: 0", "label_num"])

In [6]:
df.head()

,label,text
0,ham,Subject: enron methanol ; meter # : 988291\r\n...
1,ham,"Subject: hpl nom for january 9 , 2001\r\n( see..."
2,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar..."
3,spam,"Subject: photoshop , windows , office . cheap ..."
4,ham,Subject: re : indian springs\r\nthis deal is t...


In [7]:
df["label"].value_counts()

label
ham     3672
spam    1499
Name: count, dtype: int64

In [8]:
# Number of duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Number of unique email texts
print("Unique emails:", df["text"].nunique())
print("Total emails:", len(df))

Duplicate rows: 178
Unique emails: 4993
Total emails: 5171


## Train-Test Split

The dataset is divided into:
- 75% Training
- 25% Testing

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.25,
    random_state=42
)

# Naive Bayes from Scratch

We will:
1. Build a vocabulary.
2. Calculate prior probabilities.
3. Calculate likelihoods using Laplace smoothing.
4. Predict the class with maximum probability.

In [10]:
class NaiveBayesScratch:

    def fit(self, texts, labels):

        self.classes = np.unique(labels)

        self.class_word_counts = defaultdict(Counter)
        self.class_counts = Counter(labels)
        self.total_words = defaultdict(int)

        vocabulary = set()

        for text, label in zip(texts, labels):

            words = text.lower().split()

            vocabulary.update(words)

            self.class_word_counts[label].update(words)

            self.total_words[label] += len(words)

        self.vocab = list(vocabulary)
        self.vocab_size = len(self.vocab)

        total_docs = len(labels)

        self.priors = {
            c: self.class_counts[c] / total_docs
            for c in self.classes
        }

    def predict(self, texts):

        predictions = []

        for text in texts:

            words = text.lower().split()

            scores = {}

            for c in self.classes:

                score = np.log(self.priors[c])

                for word in words:

                    word_count = self.class_word_counts[c][word]

                    prob = (word_count + 1) / (
                        self.total_words[c] + self.vocab_size
                    )

                    score += np.log(prob)

                scores[c] = score

            predictions.append(max(scores, key=scores.get))

        return predictions

## Train the Model

In [11]:
nb = NaiveBayesScratch()

nb.fit(X_train, y_train)

predictions = nb.predict(X_test)

## Evaluate the Scratch Implementation

In [13]:
print("Scratch model Classification Report\n")
print(classification_report(y_test, predictions))

print("Confusion Matrix\n")
print(confusion_matrix(y_test, predictions))

Scratch model Classification Report

              precision    recall  f1-score   support

         ham       0.99      0.97      0.98       930
        spam       0.93      0.97      0.95       363

    accuracy                           0.97      1293
   macro avg       0.96      0.97      0.96      1293
weighted avg       0.97      0.97      0.97      1293

Confusion Matrix

[[904  26]
 [ 12 351]]


# Scikit-learn Implementation

Now we compare our implementation with the optimized library version.

In [14]:
vectorizer = CountVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)

X_test_vec = vectorizer.transform(X_test)

In [15]:
model = MultinomialNB()

model.fit(X_train_vec, y_train)

sk_predictions = model.predict(X_test_vec)

## Evaluate Scikit-learn Model

In [16]:
print("Scikit-learn Classification Report\n")

print(classification_report(y_test, sk_predictions))

print("Confusion Matrix\n")

print(confusion_matrix(y_test, sk_predictions))

Scikit-learn Classification Report

              precision    recall  f1-score   support

         ham       0.98      0.98      0.98       930
        spam       0.95      0.95      0.95       363

    accuracy                           0.97      1293
   macro avg       0.97      0.96      0.97      1293
weighted avg       0.97      0.97      0.97      1293

Confusion Matrix

[[913  17]
 [ 19 344]]


# Comparison

| Feature | Scratch | Scikit-learn |
|----------|----------|--------------|
| Built using NumPy | Yes | No |
| Uses Laplace Smoothing | Yes | Yes |
| Easy to understand | Yes | No |
| Optimized for speed | No | Yes |
| Suitable for production | No | Yes |

## Interpretation
- The **scratch** implementation **detected slightly more spam** emails:
    - Spam recall: **0.97**
    - It missed only **12 spam emails.**
- The **Scikit-learn model** produced **fewer false spam** detections:
    - Ham precision: **0.98**
    - Only **17 legitimate emails** were incorrectly marked as spam.
- Both models have **almost identical performance**, proving that the from-scratch implementation correctly follows the Naive Bayes algorithm.

# Conclusion

- Successfully implemented **Multinomial Naive Bayes** from scratch.
- Applied Laplace smoothing to avoid zero probabilities.
- Classified spam and ham emails using Bayes' theorem.
- Evaluated the model using:
  - Accuracy
  - Precision
  - Recall
  - F1-score
  - Confusion Matrix
- Compared the results with Scikit-learn's `MultinomialNB`.